# GPS Mobility Data Analysis — Yogyakarta (COVID-19)

**Visualisation notebook.** All heavy data processing lives in the `src/mobility` package and `scripts/` (DuckDB + H3). This notebook only loads prepared Parquet and produces **static** plots — no interactive maps / lonboard.

~ Dany Laksono

## Context

- Anonymous mobile-device GPS pings in **Daerah Istimewa Yogyakarta (DIY)**, Indonesia.
- Window: **23–31 Oct 2021** (~12.1M pings, ~398k devices).
- Paired with a **people graph** (device demographics + activity intensity).
- COVID-19 mobility study — interpret against the PPKM restriction timeline.

Raw CSVs cover Oct 2021 – May 2022: run `scripts/process_mobility.py` to convert them, and `scripts/prepare_spatial.py` for the H3/OD aggregates (see `AGENTS.md` / `docs/WORKFLOW.md`).

In [ ]:
# Requires the project package (editable install):  pip install -e .
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from mobility import config, io, spatial

%matplotlib inline
print("imports OK")

## 1. Load prepared data

In [ ]:
# Loads the cleaned mobility parquet (default: legacy October-2021 file).
mobility_df = io.load_mobility(config.DEFAULT_MOBILITY_PARQUET)
people      = io.load_people(config.DEFAULT_PEOPLE_PARQUET)

print(f"mobility: {len(mobility_df):,} pings, {mobility_df['maid'].nunique():,} devices")
print(f"range   : {mobility_df.datetime.min()} -> {mobility_df.datetime.max()}")
print(f"people  : {len(people):,} rows, {people.maid.nunique():,} profiles")
mobility_df[["maid", "latitude", "longitude", "datetime"]].head()

## 2. Temporal analysis

In [ ]:
daily = mobility_df.groupby("date").agg(
    pings=("maid", "size"), devices=("maid", "nunique"))
labels = [d.strftime("%d %b") for d in daily.index]

fig, axes = plt.subplots(1, 2, figsize=(15, 4.2))

bars = axes[0].bar(range(len(daily)), daily["pings"].values / 1e6, color="#4C72B0")
axes[0].set_xticks(range(len(daily))); axes[0].set_xticklabels(labels, rotation=45)
axes[0].set_title("GPS pings per day"); axes[0].set_ylabel("Pings (millions)")

bars = axes[1].bar(range(len(daily)), daily["devices"].values / 1e3, color="#DD8452")
axes[1].set_xticks(range(len(daily))); axes[1].set_xticklabels(labels, rotation=45)
axes[1].set_title("Active devices per day"); axes[1].set_ylabel("Devices (thousands)")
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4.2))

# Hourly timeline
hourly = mobility_df.set_index("datetime").resample("1h").size()
axes[0].plot(hourly.index, hourly.values / 1e3, lw=1.5, color="#55A868")
axes[0].set_title("Pings per hour"); axes[0].set_ylabel("Pings (thousands)")
axes[0].xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))

# Diurnal rhythm (mean pings per hour of day)
prof = mobility_df.groupby("hour").size() / mobility_df["date"].nunique()
axes[1].plot(prof.index, prof.values / 1e3, marker="o", ms=4, color="#4C72B0")
axes[1].set_title("Diurnal pattern (mean pings per hour)")
axes[1].set_ylabel("Pings (thousands)"); axes[1].set_xticks(range(0, 24))
plt.tight_layout(); plt.show()

## 3. People graph (device profiles)

In [ ]:
prof = people.drop_duplicates("maid")

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

sex = prof["sex"].value_counts()
axes[0].bar(sex.index, sex.values, color=["#4C72B0", "#DD8452"])
axes[0].set_title("By sex"); axes[0].tick_params(axis="x", rotation=30)

inten = prof["intensity"].value_counts()
axes[1].bar(inten.index, inten.values, color=["#55A868", "#F5C542", "#C44E52"])
axes[1].set_title("By intensity"); axes[1].tick_params(axis="x", rotation=30)

kab = prof["place1"].value_counts()
axes[2].barh(kab.index[::-1], kab.values[::-1] / 1e3, color="#4C72B0")
axes[2].set_title("By regency (kabupaten/kota)"); axes[2].set_xlabel("Devices (thousands)")
plt.tight_layout(); plt.show()

## 4. Spatial (static only)

In [ ]:
# Lightweight static density map straight from the points.
fig, ax = plt.subplots(figsize=(9, 8))
hb = ax.hexbin(mobility_df.longitude, mobility_df.latitude,
               gridsize=200, cmap="magma", bins="log", mincnt=1)
fig.colorbar(hb, ax=ax, shrink=0.8, label="log10(pings/bin)")
ax.set_aspect("equal"); ax.set_title("GPS ping density (hexbin)")
ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")
plt.tight_layout(); plt.show()

In [ ]:
# H3 grid cached by scripts/prepare_spatial.py (res=8, per day).
# If missing, run:  python scripts/prepare_spatial.py --out data/processed
grid_path = config.PROCESSED_DIR / "h3_grid_r8_daily.parquet"
if grid_path.exists():
    grid = pd.read_parquet(grid_path)
    top = grid.nlargest(50_000, "count")
    lats, lons = spatial.grid_centroids(top["h3"])
    fig, ax = plt.subplots(figsize=(9, 8))
    sc = ax.scatter(lons, lats, c=np.log10(top["count"]), s=4, cmap="magma", alpha=0.8)
    fig.colorbar(sc, ax=ax, shrink=0.8, label="log10(pings)")
    ax.set_aspect("equal")
    ax.set_title(f"H3 r8 cell density (top {len(top):,} cells)")
    ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")
    plt.tight_layout(); plt.show()
else:
    print("H3 grid not found. Run first:\n  python scripts/prepare_spatial.py --out data/processed")

## 5. Mobility metrics (per device per day)

Per-device-day metrics (`n_pings`, `n_cells`, `n_trips`, `radius_km`,
`max_dist_home_km`, `stay_at_home`) and per-device home cells are computed by
`scripts/compute_metrics.py` and cached under `data/processed/`. This section
only loads and plots them — the raw material for the glyph library.

If the files are missing, run:
`python scripts/compute_metrics.py --out data/processed --res 8`

In [ ]:
### 5.1 Load the cached metrics

metrics_path = config.PROCESSED_DIR / "metrics_device_day_r8.parquet"
homes_path   = config.PROCESSED_DIR / "homes_r8.parquet"

if metrics_path.exists():
    metrics_df = pd.read_parquet(metrics_path)
    homes_df   = pd.read_parquet(homes_path)
    print(f"metrics : {len(metrics_df):,} device-days, "
          f"{metrics_df['maid'].nunique():,} devices")
    print(f"homes   : {len(homes_df):,} devices")
    metrics_df.head()
else:
    print("Run first:\n  python scripts/compute_metrics.py --out data/processed --res 8")

In [ ]:
### 5.2 Glyph-ready distributions (static)

if metrics_path.exists():
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))

    r = metrics_df["radius_km"].dropna()
    axes[0].hist(r.clip(upper=20), bins=40, color="#4C72B0")
    axes[0].set_title("Radius of gyration (km)")
    axes[0].set_xlabel("km (clipped at 20)")

    c = metrics_df["n_cells"]
    axes[1].hist(c.clip(upper=50), bins=40, color="#55A868")
    axes[1].set_title("Distinct H3 cells visited / day")
    axes[1].set_xlabel("cells (clipped at 50)")

    sar = metrics_df.groupby("date")["stay_at_home"].mean() * 100
    axes[2].plot(sar.index, sar.values, marker="o", ms=4, color="#C44E52")
    axes[2].set_title("Stay-at-home rate over time (%)")
    axes[2].set_ylabel("% devices that never left home cell")
    plt.tight_layout(); plt.show()
else:
    print("no metrics yet — see previous cell")

### 5.3 Mobility change from baseline (index)

A daily summary with **percent change vs a baseline window** (Google/Apple
mobility-report style) is produced by `scripts/compute_metrics.py`
(`mobility_index_r<res>.parquet`, plus a by-regency variant). This cell plots
the indexed metrics.

Run with:
`python scripts/compute_metrics.py --out data/processed --res 8 --baseline-days 3`

In [ ]:
index_path = config.PROCESSED_DIR / "mobility_index_r8.parquet"
if index_path.exists():
    index_df = pd.read_parquet(index_path)
    labels = [d.strftime("%d %b") for d in index_df["date"]]

    fig, axes = plt.subplots(1, 2, figsize=(15, 4.2))

    ax = axes[0]
    ax.plot(index_df["date"], index_df["n_pings_pct_change"], marker="o", ms=4, color="#4C72B0")
    ax.axhline(0, color="grey", lw=1)
    ax.set_title("Ping volume: % change vs baseline")
    ax.set_ylabel("% vs baseline")
    ax.set_xticks(index_df["date"]); ax.set_xticklabels(labels, rotation=45)

    ax = axes[1]
    ax.plot(index_df["date"], index_df["stay_at_home_pct_change"], marker="o", ms=4, color="#C44E52")
    ax.axhline(0, color="grey", lw=1)
    ax.set_title("Stay-at-home rate: % change vs baseline")
    ax.set_ylabel("% vs baseline")
    ax.set_xticks(index_df["date"]); ax.set_xticklabels(labels, rotation=45)

    plt.tight_layout(); plt.show()
else:
    print("Run first:\n  python scripts/compute_metrics.py --out data/processed --res 8 --baseline-days 3")

## 6. Origin–destination (OD) flows

District-level flows, net flows, and per-corridor change vs baseline are
produced by `scripts/prepare_spatial.py --people ... --baseline-days N` and
cached under `data/processed/`. This section loads and plots them (the raw
material for the OD glyph library).

Run with:
`python scripts/prepare_spatial.py --people data/parquet/people.parquet --baseline-days 3 --out data/processed`

In [ ]:
### 6.1 Load district flows & corridor change

dist_path = config.PROCESSED_DIR / "od_flows_district_r8.parquet"
corr_path = config.PROCESSED_DIR / "od_corridor_change_r8.parquet"

if dist_path.exists():
    od_dist = pd.read_parquet(dist_path)
    od_corr = pd.read_parquet(corr_path)
    print(f"district flows : {len(od_dist):,} rows")
    print(f"corridor change: {len(od_corr):,} rows")
    print("\ntop corridors by total pings (all days):")
    top = (od_dist.groupby(["origin_place1", "dest_place1"])["n_pings"]
           .sum().sort_values(ascending=False).head(10))
    print(top.to_string())
    od_dist.head()
else:
    print("Run first:\n  python scripts/prepare_spatial.py --people data/parquet/people.parquet --baseline-days 3 --out data/processed")

In [ ]:
### 6.2 OD plots (static)

if dist_path.exists():
    fig, axes = plt.subplots(1, 2, figsize=(15, 4.2))

    # mean trip length over time (cell-level flows)
    odf = pd.read_parquet(config.PROCESSED_DIR / "od_flows_r8.parquet")
    mean_len = odf.groupby("date")["od_distance_km"].mean()
    axes[0].plot(mean_len.index, mean_len.values, marker="o", ms=4, color="#8172B3")
    axes[0].set_title("Mean OD trip length (km)"); axes[0].set_ylabel("km")
    axes[0].set_xticks(mean_len.index)
    axes[0].set_xticklabels([d.strftime("%d %b") for d in mean_len.index], rotation=45)

    # top district corridors over time
    top_pairs = (od_dist.groupby(["origin_place1", "dest_place1"])["n_pings"]
                 .sum().nlargest(5).index)
    for o, d in top_pairs:
        s = od_dist[(od_dist["origin_place1"] == o) & (od_dist["dest_place1"] == d)]
        axes[1].plot(s["date"], s["n_pings"], marker="o", ms=3, label=f"{o} -> {d}")
    axes[1].set_title("Top district corridors (daily pings)")
    axes[1].legend(fontsize=8)
    axes[1].set_xticks(mean_len.index)
    axes[1].set_xticklabels([d.strftime("%d %b") for d in mean_len.index], rotation=45)

    plt.tight_layout(); plt.show()
else:
    print("no OD data yet — see previous cell")

## 7. Contact / meeting index

A **co-location exposure proxy**: devices in the same H3 cell within the same
hour count as potential "meetings" (see `src/mobility/contacts.py` for the
rationale and caveats). Produced by `scripts/compute_contacts.py` and cached
under `data/processed/`.

Run with: `python scripts/compute_contacts.py --out data/processed --res 8 --bucket 1h`

In [ ]:
mi_path  = config.PROCESSED_DIR / "meeting_index_daily_r8_1h.parquet"
crow_path = config.PROCESSED_DIR / "crowding_r8_1h.parquet"

if mi_path.exists():
    mid  = pd.read_parquet(mi_path)
    crow = pd.read_parquet(crow_path)
    labels = [d.strftime("%d %b") for d in mid["date"]]

    fig, axes = plt.subplots(1, 2, figsize=(15, 4.2))

    axes[0].plot(mid["date"], mid["meeting_index"], marker="o", ms=4, color="#C44E52")
    axes[0].set_title("Daily meeting index (pairwise co-locations)")
    axes[0].set_ylabel("meetings")
    axes[0].set_xticks(mid["date"]); axes[0].set_xticklabels(labels, rotation=45)

    axes[1].plot(crow["bucket"], crow["cells_n10"], marker="o", ms=3, color="#8172B3")
    axes[1].set_title("Crowded cells (>=10 devices / hour)")
    axes[1].set_ylabel("cells"); axes[1].tick_params(axis="x", rotation=45)

    plt.tight_layout(); plt.show()
else:
    print("Run first:\n  python scripts/compute_contacts.py --out data/processed --res 8 --bucket 1h")

## 8. Spatial concentration (Gini / HHI)

Daily **Gini** and **Herfindahl (HHI)** indices of the H3 grid — do people
crowd into fewer cells during restrictions, or spread out? Produced by
`scripts/prepare_spatial.py` → `concentration_r<res>.parquet`.

Run with: `python scripts/prepare_spatial.py --out data/processed --res 8`

In [ ]:
conc_path = config.PROCESSED_DIR / "concentration_r8.parquet"
if conc_path.exists():
    conc = pd.read_parquet(conc_path)
    labels = [d.strftime("%d %b") for d in conc["date"]]

    fig, axes = plt.subplots(1, 2, figsize=(15, 4.2))

    axes[0].plot(conc["date"], conc["hhi"], marker="o", ms=4, color="#4C72B0")
    axes[0].set_title("Herfindahl index (HHI) of ping distribution")
    axes[0].set_ylabel("HHI")
    axes[0].set_xticks(conc["date"]); axes[0].set_xticklabels(labels, rotation=45)

    axes[1].plot(conc["date"], conc["gini"], marker="o", ms=4, color="#55A868")
    axes[1].set_title("Gini coefficient of ping distribution")
    axes[1].set_ylabel("Gini")
    axes[1].set_xticks(conc["date"]); axes[1].set_xticklabels(labels, rotation=45)

    plt.tight_layout(); plt.show()
else:
    print("Run first:\n  python scripts/prepare_spatial.py --out data/processed --res 8")

## Summary

- Volume grows sharply through the week (23 Oct ~75k → 29–31 Oct 2.4–3.5M pings) — likely a **coverage ramp-up**; verify against coverage, not just movement.
- Strong **diurnal rhythm** (commuter pattern); weekday/weekend differences visible on the well-covered days.
- Profile sample is **male-skewed (~3.4:1)** — not population-representative.
- ~92% of profiled devices live in **Sleman / Kota Yogyakarta / Bantul**.
- H3 grids and OD flows are produced by `scripts/prepare_spatial.py` (see `data/processed`).

Details: `FINDINGS.md`, `AGENTS.md`, `docs/`.